In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
import torch

In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Text Summarizer/samsum-train.csv')
test_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Text Summarizer/samsum-test.csv')
val_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Text Summarizer/samsum-validation.csv')

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data.sample(10)
#in the dialogues there is some extra stuff whihc may affect the Transformer badly like html tags \r\ , \n

,id,dialogue,summary
4742,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
8870,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
6554,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
12900,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
2596,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."
6422,13716070,Paul: Lucky you!\r\nJohn: ?\r\nPete: Our class...,"John, Pete and Paul's classes have been cancel..."
2452,13727976,Jasper: i miss you so much already :(\r\nKaren...,Karen will be back on Sunday. Karen and Jasper...
476,13681231,Ken: how long do you need?\r\nJude: i think ab...,Ken will wait inside as Jude needs 10 more min...
14716,13862652,"Victoria: Hey, I am in the toilet...And..\nSky...",Victoria is in a restaurant toilet and texts S...
10143,13728508,Sandra: Do u need any help with the party tomo...,Ronda does not need any help with the party to...


In [ ]:
train_data.shape
# at first we will be taking 4000 train data

(14732, 3)

In [ ]:
val_data.shape
# and 500 val data

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n = 4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n = 500, random_state=42).reset_index(drop=True)
# here we extracting the data randomly
# and the reset_index , sets the indexes from 0 again or else the indexes will also be randomized

In [ ]:
train_data.shape

(14732, 3)

### Data Preprocessing


In [ ]:
import re

def clean_data(text):
    text = str(text)
    text = re.sub(r'\r\n', ' ', text) # lines
    text = re.sub(r'\s+', ' ', text)   # spaces
    text = re.sub(r'<.*?>', ' ', text)  # html tags
    text = text.strip().lower()
    return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
train_data['dialogue'][0]

"amanda: i baked cookies. do you want some? jerry: sure! amanda: i'll bring you tomorrow :-)"

# Tokenize The Data

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# raw_data => tokenzied inputs for fine tuning
# 1. padding
# 2. max-lentgh
# 3. truncate
# "I ate on apple" = > 32 405, 111,727
# "mango" with padding 4 => 42 0 0 0
def tokenize(data):
    inputs = tokenizer(data["dialogue"] , padding="max_length", truncation=True, max_length=512)
    targets = tokenizer(data["summary"] , padding="max_length", truncation=True, max_length=150)

    inputs["labels"] = targets["input_ids"] # token ids =>  add to the input as labels
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [183, 232, 9, 10, 3, 23, 13635, 5081, 5, 103, 25, 241, 128, 58, 3, 12488, 651, 10, 417, 55, 183, 232, 9, 10, 3, 23, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
# input ids => dialogues tokens
# 1 => EOS  ,  0 => padding
# attention mask => tells the valid tokens and paddings
# labels = target => summary tokens

# Working with the Model

In [ ]:
# summarizing is NLP task as well as generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("device :",device)
model.to(device)

device : cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir = "/content/drive/MyDrive/Colab Notebooks/Text Summarizer/results",
    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy ="epoch",
    save_strategy ="epoch",

    warmup_steps = 500, # intially slow leanring then later on learning becomes faster

    # logging_dir = "/content/drive/MyDrive/Colab Notebooks/Text Summarizer/logs",
    # logging_steps = 100
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.392133,0.340548
2,0.359202,0.329997
3,0.354711,0.324783
4,0.341498,0.323600
5,0.338322,0.321620
6,0.333990,0.321082


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11052, training_loss=0.5021662798386918, metrics={'train_runtime': 4349.8047, 'train_samples_per_second': 20.321, 'train_steps_per_second': 2.541, 'total_flos': 1.1963132515713024e+16, 'train_loss': 0.5021662798386918, 'epoch': 6.0})

In [ ]:
# model load => fine-tune => save the model

In [ ]:
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/Text Summarizer/saved_summary_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/Text Summarizer/saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/Text Summarizer/saved_summary_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/Text Summarizer/saved_summary_model/tokenizer.json')

In [ ]:
# to use the saved model
model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Colab Notebooks/Text Summarizer/saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/Colab Notebooks/Text Summarizer/saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Test the core logic for summarizetion


In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=300,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai systems are becoming more capable due to advances in deep learning and access to large datasets. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.


In [ ]:
summary

'ai systems are becoming more capable due to advances in deep learning and access to large datasets. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.'

In [ ]:
sample_dialogue = """
Jake: hey are you guys free to watch the match tonight?
Mark: count me in, what time does it start?
Jake: around 8 PM. I was thinking we could order some pizzas.
Sarah: can't make it tonight guys, got a huge data structures lab due tomorrow morning and I haven't even started coding the tree traversal algorithms yet :(
Mark: rip sarah. cool jake, i'll come over to your place around 7:45. pepperoni for me!
Jake: cool see ya. good luck sarah!
"""

# Test it out
summary = summarize_dialogue(sample_dialogue)
print("Model Summary:", summary)

Model Summary: jake and sarah are going to watch the match tonight around 8 pm. mark will come over to jake's place around 7:45.
